# Delta Lake Basics — 02: Reading and Writing

## What you will learn
Delta has more read/write options than plain Parquet — some are Delta-specific
and critical for building correct pipelines.

In this notebook:
1. Write modes — overwrite vs append vs ignore vs error
2. Idempotent writes — `txnAppId` + `txnVersion`
3. `replaceWhere` — targeted partition replacement
4. Reading options — latest version vs specific version
5. Reading with filters — partition pruning in Delta
6. `insertInto` vs `writeTo` vs `save` — when to use which


In [1]:
import os, time, pathlib, shutil, random, datetime, subprocess, glob
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/delta_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('delta-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version} | Delta Lake ready")

random.seed(42)
N = 100_000
CATEGORIES = ["Electronics","Clothing","Books","Food","Sports","Furniture"]
COUNTRIES  = ["US","UK","DE","FR","JP","CA","AU","BR"]
STATUSES   = ["pending","confirmed","shipped","delivered","cancelled"]

schema = StructType([
    StructField("order_id",    LongType()),
    StructField("customer_id", LongType()),
    StructField("product",     StringType()),
    StructField("category",    StringType()),
    StructField("country",     StringType()),
    StructField("quantity",    IntegerType()),
    StructField("price",       DoubleType()),
    StructField("revenue",     DoubleType()),
    StructField("status",      StringType()),
    StructField("order_date",  DateType()),
])

base = datetime.date(2024, 1, 1)
rows = []
for i in range(N):
    qty   = random.randint(1, 10)
    price = round(random.uniform(5, 1000), 2)
    d     = base + datetime.timedelta(days=random.randint(0, 364))
    rows.append((i+1, random.randint(1,10000),
                 f"Product_{random.randint(1,200)}",
                 random.choice(CATEGORIES), random.choice(COUNTRIES),
                 qty, price, round(qty*price, 2),
                 random.choice(STATUSES), d))

df = spark.createDataFrame(rows, schema)
df.cache().count()
print(f"Dataset ready: {N:,} rows")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/09 13:36:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.0.2 | Delta Lake ready


26/04/09 13:36:45 WARN TaskSetManager: Stage 0 contains a task of very large size (1252 KiB). The maximum recommended task size is 1000 KiB.
26/04/09 13:36:47 WARN TaskSetManager: Stage 1 contains a task of very large size (1304 KiB). The maximum recommended task size is 1000 KiB.


Dataset ready: 100,000 rows


## Step 1 — Write Modes


In [2]:
TABLE = f"{DATA_DIR}/02_rw"
shutil.rmtree(TABLE, ignore_errors=True)

# overwrite — replaces entire table
df.write.format("delta").mode("overwrite").save(TABLE)
print(f"overwrite: {spark.read.format('delta').load(TABLE).count():,} rows")

# append — adds to existing table
extra = df.limit(1000)
extra.write.format("delta").mode("append").save(TABLE)
print(f"append   : {spark.read.format('delta').load(TABLE).count():,} rows  (+1000)")

# ignore — does nothing if table exists
df.limit(500).write.format("delta").mode("ignore").save(TABLE)
print(f"ignore   : {spark.read.format('delta').load(TABLE).count():,} rows  (unchanged)")

# error (default) — fails if table exists
try:
    df.write.format("delta").save(TABLE)
except Exception as e:
    print(f"error    : {type(e).__name__} raised  (table already exists)")

26/04/09 13:36:49 WARN TaskSetManager: Stage 4 contains a task of very large size (1252 KiB). The maximum recommended task size is 1000 KiB.
26/04/09 13:36:54 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/04/09 13:36:57 WARN TaskSetManager: Stage 11 contains a task of very large size (1252 KiB). The maximum recommended task size is 1000 KiB.


overwrite: 100,000 rows
append   : 101,000 rows  (+1000)
ignore   : 101,000 rows  (unchanged)
error    : AnalysisException raised  (table already exists)


## Step 2 — Idempotent Writes with txnAppId + txnVersion

In streaming or retry scenarios, the same data might be written twice.
`txnAppId` + `txnVersion` make writes idempotent — Delta ignores
duplicate writes with the same transaction ID.


In [3]:
IDEM_TABLE = f"{DATA_DIR}/02_idempotent"
shutil.rmtree(IDEM_TABLE, ignore_errors=True)

def write_batch(batch_id, df_batch):
    """Idempotent write — safe to retry without duplicating data."""
    df_batch.write.format("delta") \
            .mode("append") \
            .option("txnAppId",      "my_pipeline_v1")  # unique pipeline name
            .option("txnVersion",    str(batch_id))      # unique batch ID
            .save(IDEM_TABLE)

# First write of batch 1
write_batch(1, df.limit(1000))
print(f"After first write of batch 1: {spark.read.format('delta').load(IDEM_TABLE).count():,} rows")

# Retry batch 1 (e.g., after a failure) — Delta detects duplicate and skips
write_batch(1, df.limit(1000))
print(f"After RETRY of batch 1:       {spark.read.format('delta').load(IDEM_TABLE).count():,} rows  (same!)")

# Write new batch 2
write_batch(2, df.limit(500))
print(f"After batch 2:                {spark.read.format('delta').load(IDEM_TABLE).count():,} rows  (+500)")

IndentationError: unexpected indent (3590909552.py, line 9)

## Step 3 — replaceWhere: Targeted Overwrite

`replaceWhere` replaces only rows matching a predicate — much faster than
full table overwrite when you reprocess a specific date or partition.


In [ ]:
REPLACE_TABLE = f"{DATA_DIR}/02_replace"
shutil.rmtree(REPLACE_TABLE, ignore_errors=True)

# Initial full table
df.write.format("delta").mode("overwrite").save(REPLACE_TABLE)
initial = spark.read.format("delta").load(REPLACE_TABLE).count()
print(f"Initial table: {initial:,} rows")

# Count Electronics before
elec_before = spark.read.format("delta").load(REPLACE_TABLE) \
                   .filter(F.col("category")=="Electronics").count()
print(f"Electronics rows before: {elec_before:,}")

# Reprocess only Electronics — overwrite just those rows
new_electronics = df.filter(F.col("category")=="Electronics") \
                    .withColumn("price", F.col("price") * 1.1)  # price increase

new_electronics.write.format("delta") \
               .mode("overwrite") \
               .option("replaceWhere", "category = 'Electronics'") \
               .save(REPLACE_TABLE)

after = spark.read.format("delta").load(REPLACE_TABLE).count()
elec_after = spark.read.format("delta").load(REPLACE_TABLE) \
                  .filter(F.col("category")=="Electronics").count()

print(f"After replaceWhere: {after:,} rows (same total)")
print(f"Electronics rows after: {elec_after:,} (same count, different prices)")
print()
print("DeltaTable history:")
DeltaTable.forPath(spark, REPLACE_TABLE).history() \
    .select("version","operation","operationParameters").show(truncate=False)

## Step 4 — Reading Options


In [ ]:
# Latest version (default)
df_latest = spark.read.format("delta").load(REPLACE_TABLE)
print(f"Latest version: {df_latest.count():,} rows")

# Specific version
df_v0 = spark.read.format("delta").option("versionAsOf", 0).load(REPLACE_TABLE)
print(f"Version 0     : {df_v0.count():,} rows")

# Timestamp-based read
from datetime import datetime, timedelta
ts = (datetime.now() - timedelta(seconds=5)).strftime("%Y-%m-%d %H:%M:%S")
df_ts = spark.read.format("delta").option("timestampAsOf", ts).load(REPLACE_TABLE)
print(f"5 seconds ago : {df_ts.count():,} rows")

# Partition filter (partition pruning)
df_elec = (spark.read.format("delta").load(REPLACE_TABLE)
           .filter(F.col("category") == "Electronics"))
print(f"\nElectronics only: {df_elec.count():,} rows")
print("\nExplain — check for partition filters:")
df_elec.explain(mode="simple")

## Step 5 — insertInto vs writeTo vs save

Three APIs for writing to Delta — each has different semantics:


In [ ]:
# Register as temp view for SQL
df.createOrReplaceTempView("source_data")

# Create catalog table
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS delta.`{DATA_DIR}/02_catalog`
    USING delta
    AS SELECT * FROM source_data LIMIT 1000
""")

print("save()     → path-based write, most flexible")
print("writeTo()  → catalog-based, type-safe column matching")
print("insertInto()→ catalog-based, position-based column matching")
print()
print("Recommendation: use save() for path-based tables (our setup)")
print("Use writeTo() / insertInto() when tables are registered in a catalog")
print()

# save() — path based
df.limit(100).write.format("delta").mode("append").save(f"{DATA_DIR}/02_save_demo")
print(f"save(): {spark.read.format('delta').load(f'{DATA_DIR}/02_save_demo').count()} rows")

## Summary

```python
# Standard write
df.write.format("delta").mode("overwrite").save(path)
df.write.format("delta").mode("append").save(path)

# Idempotent write (safe retries)
df.write.format("delta").mode("append")
        .option("txnAppId", "pipeline_v1")
        .option("txnVersion", str(batch_id))
        .save(path)

# Targeted overwrite (replace only matching rows)
df.write.format("delta").mode("overwrite")
        .option("replaceWhere", "category = 'Electronics'")
        .save(path)

# Read options
spark.read.format("delta").load(path)                              # latest
spark.read.format("delta").option("versionAsOf", 3).load(path)    # version
spark.read.format("delta").option("timestampAsOf", ts).load(path) # timestamp
```
